In [ ]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 224 (delta 99), reused 181 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 777.80 KiB | 4.71 MiB/s, done.
Resolving deltas: 100% (99/99), done.


In [ ]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 437.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 351.1 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [ ]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 60.9 MB/s eta 0:00:00


In [ ]:
from typing import Optional
from enum import Enum
from logging import Logger

import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (MultiAttemptSuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SeeMoreBlocksRewardModel(RewardModel):
    def __init__(self):
        self._blocks_seen = set()

    def reset(self) -> None:
        self._blocks_seen = set()

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    return 1.0
        return 0.0

In [ ]:
RUN_NAME = "tile_finder"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
ATTEMPTS = 3

In [ ]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [ ]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = SeeMoreBlocksRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
env = SuperMarioDiscretizer(base_env)
encoder = TileEncoder()

2026-05-01 01:51:42 [INFO] Session log for run tile_finder with level [INFO] initialized at: tile_finder_20260501_015142.log
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
try:
    distinct_observations = []
    last_distinct_obs = None
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(SuperMarioCombo.get_combo_id(SuperMarioCombo.RIGHT))
        if last_distinct_obs is None or not np.array_equal(obs, last_distinct_obs):
            last_distinct_obs = obs
            distinct_observations.append(obs)
        done = terminated or truncated
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
encoder.train(distinct_observations, print_progress=True)

Epoch 5/5: 100%|██████████| 5/5 [00:00<00:00, 63.52it/s, loss=5.89]


In [ ]:
class ApplyEncodingWrapper(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, encoder: TileEncoder, device: Optional[str] = None):
        super().__init__(env)
        self._encoder = encoder
        self.device = device
        if self.device is not None:
            self._encoder.to(device)
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(16, 14, 16),
            dtype=np.uint8
        )

    def observation(self, observation: np.ndarray) -> np.ndarray:
        features = self._encoder.embed(observation)
        features = features.permute(2, 0, 1).cpu().numpy()
        return (features * 255).astype(np.uint8)

In [ ]:
try:
    encoded_env = ApplyEncodingWrapper(env, encoder)
    ppo_env = DummyVecEnv([lambda: encoded_env])
    model = PPO("MlpPolicy", ppo_env, verbose=1, learning_rate=0.0003, n_steps=10000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger)
    logger.info("Starting training...")
    model.learn(total_timesteps=60000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/preprocessing.py:22: UserWarning: Treating image space as channels-last, while second dimension was smallest of the three.
  warnings.warn("Treating image space as channels-last, while second dimension was smallest of the three.")
/usr/local/lib/python3.12/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 10000`, after every 156 untruncated mini-batches, there will be a truncated mini-batch of size 16
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=10000 and n_envs=1)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See 

Using cuda device
Wrapping the env in a VecTransposeImage.


2026-05-01 01:52:19 [INFO] Priming complete
2026-05-01 01:52:19 [INFO] Starting training...
2026-05-01 01:53:09 [INFO] Step 10000 - time/iterations: 1
2026-05-01 01:53:09 [INFO] Step 10000 - time/fps: 197
2026-05-01 01:53:09 [INFO] Step 10000 - time/time_elapsed: 50
2026-05-01 01:53:09 [INFO] Step 10000 - time/total_timesteps: 10000
2026-05-01 01:54:08 [INFO] Step 20000 - train/learning_rate: 0.0003
2026-05-01 01:54:08 [INFO] Step 20000 - train/entropy_loss: -0.045946490095489345
2026-05-01 01:54:08 [INFO] Step 20000 - train/policy_gradient_loss: -8.227470194458203e-05
2026-05-01 01:54:08 [INFO] Step 20000 - train/value_loss: 0.1747081811925408
2026-05-01 01:54:08 [INFO] Step 20000 - train/approx_kl: 8.804899698588997e-05
2026-05-01 01:54:08 [INFO] Step 20000 - train/clip_fraction: 0.0014530254777070063
2026-05-01 01:54:08 [INFO] Step 20000 - train/loss: 0.10117775201797485
2026-05-01 01:54:08 [INFO] Step 20000 - train/explained_variance: 0.008980035781860352
2026-05-01 01:54:08 [INFO]

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    encoded_env = ApplyEncodingWrapper(env, encoder)
    video_env = RecordVideo(encoded_env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()
#     del env
#     import gc
#     gc.collect()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-01 01:58:17 [INFO] Step: 1000
2026-05-01 01:58:18 [INFO] Terminated: True
2026-05-01 01:58:18 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
